# 04 · Experiment A (baselines) and Experiment B (ablation)
**A — Baseline comparison:** ResNet, DenseNet, EfficientNet (2.5D slice-wise + max/mean pooling), a plain 3D CNN, a 3D Vision Transformer, and the proposed model.
**B — Ablation:** cumulative `baseline → + vessel → + multi-scale → + topology attention → + multi-task → + FP suppression`.

All models use the same splits, optimiser, schedule and detection loss. Bootstrap 95% CIs are reported on the **test** set.
For the manuscript, repeat each run with ≥3 seeds (`SEEDS`) and report mean ± SD.

In [ ]:
%run ./00_common.ipynb

In [ ]:
import torchvision.models as tvm, pandas as pd
sp = load_splits(); tr, va, te = sp["train"], sp["val"], sp["test"]
SEEDS = [42]; EPOCHS = cfg.epochs

class Slice2D(nn.Module):
    """2.5D baseline: S evenly spaced slices (z-1,z,z+1 as RGB) -> ImageNet-style CNN -> mean+max over slices -> logit."""
    def __init__(s, name="resnet18", n_slices=24, pretrained=False):
        super().__init__()
        m = getattr(tvm, name)(weights="DEFAULT" if pretrained else None)
        if hasattr(m, "fc"): dim = m.fc.in_features; m.fc = nn.Identity()
        else:
            cl = m.classifier; dim = (cl[-1] if isinstance(cl, nn.Sequential) else cl).in_features; m.classifier = nn.Identity()
        s.m, s.S = m, n_slices; s.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(2 * dim, 1))
    def forward(s, b):
        x = b["vol"][:, 0]; B, D = x.shape[:2]
        idx = torch.linspace(1, D - 2, s.S, device=x.device).long()
        t = torch.stack([x[:, idx - 1], x[:, idx], x[:, idx + 1]], 2)              # (B,S,3,H,W)
        f = s.m(t.flatten(0, 1)).view(B, s.S, -1)
        return dict(det_logit=s.head(torch.cat([f.mean(1), f.amax(1)], 1)).squeeze(-1))

class Simple3D(nn.Module):
    def __init__(s, ch=(16, 32, 64, 128)):
        super().__init__(); L, i = [], 1
        for k, c in enumerate(ch): L.append(conv_block(i, c, 1 if k == 0 else 2)); i = c
        s.enc = nn.Sequential(*L); s.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(2 * ch[-1], 1))
    def forward(s, b):
        f = s.enc(b["vol"]); return dict(det_logit=s.head(torch.cat([f.mean((2, 3, 4)), f.amax((2, 3, 4))], 1)).squeeze(-1))

class ViT3D(nn.Module):
    def __init__(s, shape=(64, 128, 128), patch=(8, 16, 16), C=128, depth=4, heads=4):
        super().__init__()
        s.pe = nn.Conv3d(1, C, patch, patch); n = int(np.prod([a // p for a, p in zip(shape, patch)]))
        s.cls = nn.Parameter(torch.zeros(1, 1, C)); s.pos = nn.Parameter(torch.zeros(1, n + 1, C)); nn.init.trunc_normal_(s.pos, std=.02)
        s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(C, heads, 2 * C, 0.1, batch_first=True, norm_first=True), depth)
        s.norm = nn.LayerNorm(C); s.head = nn.Linear(C, 1)
    def forward(s, b):
        t = s.pe(b["vol"]).flatten(2).transpose(1, 2)
        t = torch.cat([s.cls.expand(len(t), -1, -1), t], 1) + s.pos
        return dict(det_logit=s.head(s.norm(s.enc(t))[:, 0]).squeeze(-1))

def boot_ci(y, p, fn, n=500, seed=0):
    r = np.random.RandomState(seed); v = []
    for _ in range(n):
        i = r.randint(0, len(y), len(y))
        if len(np.unique(y[i])) > 1: v.append(fn(y[i], p[i]))
    return np.percentile(v, [2.5, 97.5])

def run_exp(name, build, seed):
    set_seed(seed); model = build()
    model, hist, best = train_model(model, tr, va, cfg, EPOCHS, tag=f"{name}_s{seed}", log=lambda *_: None)
    R = predict(model, make_loader(te, cfg, False)); y, p = R["y"][:, 0], R["det_prob"]
    lo, hi = boot_ci(y, p, safe_auc)
    row = dict(model=name, seed=seed, params_M=round(count_params(model), 2), val_auc=round(best, 4), test_auc=round(safe_auc(y, p), 4),
               auc_lo=round(lo, 4), auc_hi=round(hi, 4), test_ap=round(float(average_precision_score(y, p)), 4))
    if "slice_prob" in R:
        thr = json.load(open(f"{cfg.ckpt_dir}/thresholds.json"))["thr_slice"] if os.path.exists(f"{cfg.ckpt_dir}/thresholds.json") else 0.5
        row["slice_f1"] = round(slice_f1(R["slice_prob"], R["slice_lab"], R["slice_valid"], thr), 4)
        row["fp_lesions_per_neg"] = round(fp_lesions_per_negative(R["slice_prob"], y, thr), 3)
    print(row); return row

## Experiment A — baseline comparison

In [ ]:
BASELINES = {
    "ResNet-18 (2.5D)":       lambda: Slice2D("resnet18"),
    "DenseNet-121 (2.5D)":    lambda: Slice2D("densenet121"),
    "EfficientNet-B0 (2.5D)": lambda: Slice2D("efficientnet_b0"),
    "3D CNN":                 lambda: Simple3D(),
    "3D ViT":                 lambda: ViT3D(cfg.vol_shape),
    "Proposed (full)":        lambda: PENet(cfg, Flags()),
}
rowsA = [run_exp(n, bld, s) for n, bld in BASELINES.items() for s in SEEDS]
A = pd.DataFrame(rowsA); A.to_csv(f"{cfg.out_dir}/expA_baselines.csv", index=False); A

## Experiment B — cumulative ablation

In [ ]:
STEPS = [
    ("Baseline (3D encoder)",       Flags(False, False, False, False, False)),
    ("+ Vessel representation",    Flags(True,  False, False, False, False)),
    ("+ Multi-scale features",      Flags(True,  True,  False, False, False)),
    ("+ Topology attention",        Flags(True,  True,  True,  False, False)),
    ("+ Multi-task learning",       Flags(True,  True,  True,  True,  False)),
    ("+ FP suppression (full)",     Flags(True,  True,  True,  True,  True)),
]
rowsB = [run_exp(n, (lambda f=f: PENet(cfg, f)), s) for n, f in STEPS for s in SEEDS]
B = pd.DataFrame(rowsB); B.to_csv(f"{cfg.out_dir}/expB_ablation.csv", index=False); B

In [ ]:
import matplotlib.pyplot as plt
g = B.groupby("model", sort=False).agg(auc=("test_auc", "mean"), lo=("auc_lo", "mean"), hi=("auc_hi", "mean")).reset_index()
plt.figure(figsize=(8, 3.5)); plt.errorbar(range(len(g)), g.auc, yerr=[g.auc - g.lo, g.hi - g.auc], fmt="o-", capsize=4)
plt.xticks(range(len(g)), g.model, rotation=25, ha="right"); plt.ylabel("test AUC (95% bootstrap CI)"); plt.title("Ablation"); plt.tight_layout(); plt.show()